<a href="https://colab.research.google.com/github/YefridC09/ST-554-Project1-Template/blob/main/Task3/Task_3_Project1_ST_554.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3 - Project 1

## Author: Yefrid Cordoba

### Importing the data

First install the repo where the data is stored

In [36]:
!pip install ucimlrepo

From the repo, we extract the list of air quality for our analysis, including filtering not existing data for C6H6(GT), CO(GT), T, RH, or AH

In [37]:
import ucimlrepo as uci
import numpy as np
import pandas as pd
import sklearn as sk


air_quality = uci.fetch_ucirepo(id=360)
air_quality = air_quality.data.features
air_quality = air_quality[["Date", "Time", "C6H6(GT)",
                           "CO(GT)", "T", "RH", "AH"]]
air_quality = air_quality.loc[
    (air_quality["C6H6(GT)"] != -200) &
    (air_quality["CO(GT)"] != -200) &
    (air_quality["T"] != -200) &
    (air_quality["RH"] != -200) &
    (air_quality["AH"] != -200)
]

air_quality.head()

,Date,Time,C6H6(GT),CO(GT),T,RH,AH
0,3/10/2004,18:00:00,11.9,2.6,13.6,48.9,0.7578
1,3/10/2004,19:00:00,9.4,2.0,13.3,47.7,0.7255
2,3/10/2004,20:00:00,9.0,2.2,11.9,54.0,0.7502
3,3/10/2004,21:00:00,9.2,2.2,11.0,60.0,0.7867
4,3/10/2004,22:00:00,6.5,1.6,11.2,59.6,0.7888


First we change the column date to a `pd.datetime` type object to ensure the dates are sorted in the proper way.\
Then it is grouped by day (Column `"Date"`).\
The average for each variable is calculated per day.

In [38]:
air_quality["Date"] = pd.to_datetime(air_quality["Date"])
air_quality = air_quality.groupby("Date")[["C6H6(GT)", "CO(GT)",
                                           "T", "RH", "AH"]].mean()
air_quality.head()

,C6H6(GT),CO(GT),T,RH,AH
Date,,,,,
2004-03-10,8.450000,1.966667,12.033333,54.900000,0.765633
2004-03-11,8.269565,2.239130,9.826087,64.230435,0.777039
2004-03-12,12.177273,2.804545,11.618182,50.190909,0.665164
2004-03-13,11.121739,2.695652,13.121739,50.682609,0.733013
2004-03-14,9.830435,2.469565,16.182609,48.317391,0.849209


It is added a halper column to give index for the days

In [39]:
air_quality["Day"] = range(1, len(air_quality) + 1)
air_quality.head()



,C6H6(GT),CO(GT),T,RH,AH,Day
Date,,,,,,
2004-03-10,8.450000,1.966667,12.033333,54.900000,0.765633,1
2004-03-11,8.269565,2.239130,9.826087,64.230435,0.777039,2
2004-03-12,12.177273,2.804545,11.618182,50.190909,0.665164,3
2004-03-13,11.121739,2.695652,13.121739,50.682609,0.733013,4
2004-03-14,9.830435,2.469565,16.182609,48.317391,0.849209,5


In [40]:
SLR = sk.linear_model.LinearRegression()
SLR.fit(air_quality[["CO(GT)"]], air_quality["C6H6(GT)"])
print(SLR.intercept_, SLR.coef_)

0.644770948344199 [4.56536116]


In [41]:
type(air_quality['C6H6(GT)'])

pandas.core.series.Series

In [70]:
D = 250
X_train, X_test, Y_train, Y_test = sk.model_selection.train_test_split(air_quality[["CO(GT)", "T", "RH", "AH", "Day"]], air_quality["C6H6(GT)"], train_size=D, shuffle= False)

In [73]:
X_train.tail()

,CO(GT),T,RH,AH,Day
Date,,,,,
2004-12-17,2.550000,12.925000,72.900000,1.082725,246
2004-12-18,1.762500,12.333333,44.891667,0.620742,247
2004-12-19,2.730435,5.295652,63.452174,0.568248,248
2004-12-20,2.358333,8.704167,53.708333,0.608042,249
2004-12-21,2.033333,6.683333,38.654167,0.380229,250


Defining a function that takes a dataframe with the predictors, a series witht he response variable and the day until is going to be used to train the model, then this regression is going to be used to predict the next day and the mean squared error **(MSE)** is going to be calculated based on the predicted and the measured value.

In [87]:
X_train = air_quality.iloc[D]
print(X_train)

C6H6(GT)      8.069565
CO(GT)        2.221739
T             6.682609
RH           40.600000
AH            0.395265
Day         251.000000
Name: 2004-12-22 00:00:00, dtype: float64


In [88]:
def MSE(X , Y: pd.Series, Day: int):
    """
    This function calculates the mean squared error for a given day.
    X: is a dataframe with the predictors
    Y: is a series with the target variable
    Day: is the day until the mean squared error is calculated
    (including this day).
    """
    X_train = X.iloc[:Day]
    Y_train = Y.iloc[:Day]
    X_test = X.iloc[Day]
    Y_test = Y.iloc[Day]
    reg = sk.linear_model.LinearRegression()
    reg.fit(X_train, Y_train)
    Y_pred = reg.predict(X_test.to_numpy().reshape(1, -1))
    return ((Y_test - Y_pred)**2)/Day

In [ ]:
    X_train, X_test, Y_train, Y_test = sk.model_selection.train_test_split(
        X, Y, train_size = Day, shuffle= False)
    reg = sk.linear_model.LinearRegression()
    reg.fit(X_train, Y_train)

In [82]:
air_quality.Day[250]

/tmp/ipython-input-1193432141.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  air_quality.Day[250]


np.int64(251)